In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

BASE_DIR    = Path().resolve().parent
TREC_DIR    = BASE_DIR / 'models' / 'trec'
FIGURES_DIR = BASE_DIR / 'figures'

ALGO_COLORS = {
    'Naive Bayes':         '#1C2B4A',
    'SVM':                 '#0891B2',
    'Random Forest':       '#2563EB',
    'Regressão Logística': '#64748B',
}

print(f'Projeto em : {BASE_DIR}')

In [ ]:
def load(name):
    with open(TREC_DIR / f'{name}.pkl', 'rb') as f:
        return pickle.load(f)

X_train = load('X_train_tfidf')
X_test  = load('X_test_tfidf')
y_train = load('y_train')
y_test  = load('y_test')

print(f'Treino : {X_train.shape}')
print(f'Teste  : {X_test.shape}')
print(f'\nDistribuição no teste:')
print(pd.Series(y_test).value_counts().rename({1: 'spam', 0: 'ham'}))
print(f'\nProporção spam no teste: {y_test.mean():.2%}')

In [ ]:
models = {
    'Naive Bayes':         MultinomialNB(),
    'SVM':                 LinearSVC(random_state=42, max_iter=2000, dual='auto'),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Regressão Logística': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
}

results = []
for name, model in models.items():
    print(f'Treinando: {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
        'Algoritmo': name,
        'Acurácia':  accuracy_score(y_test, y_pred),
        'Precisão':  precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        'F1-score':  f1_score(y_test, y_pred, pos_label=1, zero_division=0),
    })
    slug = name.lower().replace(' ', '_').replace('ã', 'a').replace('ç', 'c')
    with open(TREC_DIR / f'model_{slug}.pkl', 'wb') as f:
        pickle.dump(model, f)

df_results = pd.DataFrame(results).set_index('Algoritmo')
df_results = df_results.sort_values('F1-score', ascending=False)

print('\n' + '=' * 60)
print('TABELA COMPARATIVA DE DESEMPENHO — TREC 2007 (150 palavras)')
print('=' * 60)
print(df_results.round(4).to_string())
print('=' * 60)

df_results.to_csv(TREC_DIR / 'results.csv')
print('Salvo: models/trec/results.csv')

In [ ]:
print('Validação cruzada 5-fold (F1-score no treino):\n')

cv_results = {}
for name, model in models.items():
    m = model.__class__(**model.get_params())
    scores = cross_val_score(m, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<25} média={scores.mean():.4f}  std={scores.std():.4f}')

In [5]:
# Célula 5 — Relatórios de classificação detalhados
for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f'\n{"=" * 50}')
    print(f'  {name}')
    print(f'{"=" * 50}')
    print(classification_report(
        y_test, y_pred,
        target_names=['Ham', 'Spam'],
        digits=4
    ))


  Naive Bayes
              precision    recall  f1-score   support

         Ham     0.8849    0.9907    0.9348      5044
        Spam     0.9948    0.9331    0.9630      9711

    accuracy                         0.9528     14755
   macro avg     0.9399    0.9619    0.9489     14755
weighted avg     0.9573    0.9528    0.9533     14755


  SVM
              precision    recall  f1-score   support

         Ham     0.9916    0.9869    0.9893      5044
        Spam     0.9932    0.9957    0.9944      9711

    accuracy                         0.9927     14755
   macro avg     0.9924    0.9913    0.9919     14755
weighted avg     0.9927    0.9927    0.9927     14755




  Random Forest


              precision    recall  f1-score   support

         Ham     0.9926    0.9869    0.9898      5044
        Spam     0.9932    0.9962    0.9947      9711

    accuracy                         0.9930     14755
   macro avg     0.9929    0.9916    0.9922     14755
weighted avg     0.9930    0.9930    0.9930     14755


  Regressão Logística
              precision    recall  f1-score   support

         Ham     0.9904    0.9784    0.9843      5044
        Spam     0.9888    0.9951    0.9919      9711

    accuracy                         0.9894     14755
   macro avg     0.9896    0.9867    0.9881     14755
weighted avg     0.9894    0.9894    0.9893     14755



In [ ]:
metrics = ['Acurácia', 'Precisão', 'Recall', 'F1-score']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle(
    'Desempenho Comparativo dos Algoritmos — TREC 2007 (recorte 150 palavras)',
    fontsize=13, fontweight='bold', y=1.02
)

for ax, metric in zip(axes, metrics):
    valores = df_results[metric].sort_values(ascending=True)
    cores   = [ALGO_COLORS[a] for a in valores.index]
    bars = ax.barh(valores.index, valores.values, color=cores, edgecolor='white')
    ax.set_xlim(max(0, valores.min() - 0.05), 1.0)
    ax.set_title(metric, fontweight='bold', fontsize=11)
    ax.set_xlabel('Score')
    ax.tick_params(axis='y', labelsize=9)
    for bar, val in zip(bars, valores.values):
        ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'comparative_metrics_trec.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figura salva: figures/comparative_metrics_trec.png')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle(
    'Matrizes de Confusão — TREC 2007 (recorte 150 palavras)',
    fontsize=13, fontweight='bold', y=1.02
)

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Ham', 'Spam'],
        yticklabels=['Ham', 'Spam'],
        linewidths=0.5, linecolor='white'
    )
    ax.set_title(name, fontweight='bold', fontsize=10)
    ax.set_xlabel('Previsto')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrices_trec.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figura salva: figures/confusion_matrices_trec.png')

In [ ]:
melhor      = df_results['F1-score'].idxmax()
model_best  = models[melhor]
y_pred_best = model_best.predict(X_test)

fp = ((y_pred_best == 1) & (y_test == 0)).sum()
fn = ((y_pred_best == 0) & (y_test == 1)).sum()

print(f'Melhor algoritmo por F1-score: {melhor}')
print(f'\nMétricas:')
print(df_results.loc[melhor].round(4).to_string())
print(f'\nAnálise de erros ({melhor}):')
print(f'  Falsos positivos (ham → spam): {fp}')
print(f'  Falsos negativos (spam → ham): {fn}')
# Falsos negativos são mais custosos: spam não detectado chega ao usuário
print(f'\n  Falsos negativos são mais custosos em segurança —')
print(f'  representam spam que passou pelo filtro.')